# OSM Map Downloader - Indonesia

Script komprehensif untuk mengunduh data OpenStreetMap (OSM) dengan dua opsi input:
1. **SHP File (Shapefile)** - Menggunakan file shapefile sebagai batas wilayah
2. **Nama Kabupaten/Kota** - Menggunakan nama administratif wilayah Indonesia

## Fitur:
- Download berbagai layer OSM (jalan, bangunan, sungai, POI, dll)
- Export ke berbagai format (GeoJSON, Shapefile, GeoPackage)
- Visualisasi interaktif dengan Folium
- Support batch download untuk multiple wilayah
- Progress tracking dan error handling

## 1. Install Dependencies

In [ ]:
# Install required packages
!pip install osmnx geopandas folium shapely requests pandas matplotlib contextily pyproj fiona -q

## 2. Import Libraries

In [ ]:
import os
import json
import time
import warnings
from pathlib import Path
from typing import Union, List, Dict, Optional, Tuple
from dataclasses import dataclass
from enum import Enum

import osmnx as ox
import geopandas as gpd
import pandas as pd
import numpy as np
import folium
from folium.plugins import MarkerCluster
import requests
from shapely.geometry import box, Polygon, MultiPolygon, shape
from shapely.ops import unary_union
import matplotlib.pyplot as plt
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

warnings.filterwarnings('ignore')

# Configure OSMnx
ox.settings.log_console = False
ox.settings.use_cache = True
ox.settings.cache_folder = './osm_cache'

print("Libraries imported successfully!")
print(f"OSMnx version: {ox.__version__}")
print(f"GeoPandas version: {gpd.__version__}")

## 3. Konfigurasi dan Konstanta

In [ ]:
class OSMLayerType(Enum):
    """Enum untuk jenis layer OSM yang tersedia"""
    ROADS = "roads"
    BUILDINGS = "buildings"
    WATER = "water"
    LANDUSE = "landuse"
    POI = "poi"
    NATURAL = "natural"
    BOUNDARY = "boundary"
    RAILWAY = "railway"
    AMENITY = "amenity"
    ALL = "all"


class ExportFormat(Enum):
    """Enum untuk format export"""
    GEOJSON = "geojson"
    SHAPEFILE = "shp"
    GEOPACKAGE = "gpkg"
    CSV = "csv"


@dataclass
class OSMConfig:
    """Konfigurasi untuk download OSM"""
    output_dir: str = "./osm_output"
    cache_dir: str = "./osm_cache"
    timeout: int = 180
    max_retries: int = 3
    retry_delay: int = 5


# Tags OSM untuk setiap layer
OSM_TAGS = {
    OSMLayerType.ROADS: {
        'highway': True
    },
    OSMLayerType.BUILDINGS: {
        'building': True
    },
    OSMLayerType.WATER: {
        'water': True,
        'waterway': True,
        'natural': ['water', 'bay', 'strait']
    },
    OSMLayerType.LANDUSE: {
        'landuse': True
    },
    OSMLayerType.POI: {
        'amenity': True,
        'shop': True,
        'tourism': True,
        'leisure': True
    },
    OSMLayerType.NATURAL: {
        'natural': True
    },
    OSMLayerType.BOUNDARY: {
        'boundary': 'administrative'
    },
    OSMLayerType.RAILWAY: {
        'railway': True
    },
    OSMLayerType.AMENITY: {
        'amenity': True
    }
}

# Road classification untuk analisis
ROAD_CLASSIFICATION = {
    'primary': ['motorway', 'trunk', 'primary'],
    'secondary': ['secondary', 'tertiary'],
    'local': ['residential', 'living_street', 'unclassified'],
    'path': ['footway', 'cycleway', 'path', 'pedestrian']
}

print("Configuration loaded!")

## 4. Class Utama: OSMDownloader

In [ ]:
class OSMDownloader:
    """
    Class utama untuk mengunduh data OSM
    
    Mendukung input dari:
    - Shapefile (.shp)
    - Nama wilayah (kabupaten/kota)
    - Koordinat bounding box
    - GeoJSON
    """
    
    def __init__(self, config: Optional[OSMConfig] = None):
        """Initialize OSM Downloader"""
        self.config = config or OSMConfig()
        self.boundary_gdf = None
        self.boundary_polygon = None
        self.area_name = None
        self.downloaded_data = {}
        
        # Create output directories
        Path(self.config.output_dir).mkdir(parents=True, exist_ok=True)
        Path(self.config.cache_dir).mkdir(parents=True, exist_ok=True)
        
        # Configure OSMnx
        ox.settings.cache_folder = self.config.cache_dir
        ox.settings.timeout = self.config.timeout
    
    # ==================== INPUT METHODS ====================
    
    def load_from_shapefile(self, shp_path: str, 
                           dissolve: bool = True,
                           buffer_km: float = 0) -> gpd.GeoDataFrame:
        """
        Load boundary dari file shapefile
        
        Parameters:
        -----------
        shp_path : str
            Path ke file shapefile
        dissolve : bool
            Apakah menggabungkan semua polygon menjadi satu
        buffer_km : float
            Buffer dalam kilometer (opsional)
            
        Returns:
        --------
        GeoDataFrame dengan boundary
        """
        print(f"Loading shapefile: {shp_path}")
        
        if not os.path.exists(shp_path):
            raise FileNotFoundError(f"Shapefile tidak ditemukan: {shp_path}")
        
        # Read shapefile
        gdf = gpd.read_file(shp_path)
        
        # Ensure CRS is WGS84
        if gdf.crs is None:
            gdf.set_crs(epsg=4326, inplace=True)
        elif gdf.crs.to_epsg() != 4326:
            gdf = gdf.to_crs(epsg=4326)
        
        # Dissolve if requested
        if dissolve and len(gdf) > 1:
            gdf['dissolve_field'] = 1
            gdf = gdf.dissolve(by='dissolve_field').reset_index(drop=True)
        
        # Apply buffer if requested
        if buffer_km > 0:
            # Convert to projected CRS for accurate buffer
            gdf_proj = gdf.to_crs(epsg=3857)
            gdf_proj['geometry'] = gdf_proj.geometry.buffer(buffer_km * 1000)
            gdf = gdf_proj.to_crs(epsg=4326)
        
        self.boundary_gdf = gdf
        self.boundary_polygon = gdf.geometry.unary_union
        self.area_name = Path(shp_path).stem
        
        print(f"Shapefile loaded successfully!")
        print(f"  - Area: {self._calculate_area_km2():.2f} km²")
        print(f"  - Bounds: {self.boundary_gdf.total_bounds}")
        
        return self.boundary_gdf
    
    def load_from_place_name(self, place_name: str,
                            admin_level: Optional[int] = None,
                            buffer_km: float = 0) -> gpd.GeoDataFrame:
        """
        Load boundary dari nama tempat (kabupaten/kota)
        
        Parameters:
        -----------
        place_name : str
            Nama wilayah (contoh: "Kota Bandung, Indonesia")
        admin_level : int, optional
            Level administratif OSM (4=provinsi, 5=kabupaten/kota)
        buffer_km : float
            Buffer dalam kilometer
            
        Returns:
        --------
        GeoDataFrame dengan boundary
        """
        print(f"Searching for: {place_name}")
        
        try:
            # Try using OSMnx geocode_to_gdf
            gdf = ox.geocode_to_gdf(place_name)
            
            if gdf.empty:
                raise ValueError(f"Wilayah tidak ditemukan: {place_name}")
            
            # Ensure CRS
            if gdf.crs is None or gdf.crs.to_epsg() != 4326:
                gdf = gdf.to_crs(epsg=4326)
            
            # Apply buffer if requested
            if buffer_km > 0:
                gdf_proj = gdf.to_crs(epsg=3857)
                gdf_proj['geometry'] = gdf_proj.geometry.buffer(buffer_km * 1000)
                gdf = gdf_proj.to_crs(epsg=4326)
            
            self.boundary_gdf = gdf
            self.boundary_polygon = gdf.geometry.unary_union
            self.area_name = place_name.replace(',', '_').replace(' ', '_')
            
            print(f"Place found successfully!")
            print(f"  - Display name: {gdf['display_name'].iloc[0] if 'display_name' in gdf.columns else place_name}")
            print(f"  - Area: {self._calculate_area_km2():.2f} km²")
            print(f"  - Bounds: {self.boundary_gdf.total_bounds}")
            
            return self.boundary_gdf
            
        except Exception as e:
            print(f"Error with OSMnx geocode: {e}")
            print("Trying alternative method with Nominatim...")
            return self._load_from_nominatim(place_name, buffer_km)
    
    def _load_from_nominatim(self, place_name: str, buffer_km: float = 0) -> gpd.GeoDataFrame:
        """Alternative method using Nominatim API directly"""
        
        nominatim_url = "https://nominatim.openstreetmap.org/search"
        params = {
            'q': place_name,
            'format': 'json',
            'polygon_geojson': 1,
            'limit': 1
        }
        headers = {'User-Agent': 'OSMDownloader/1.0'}
        
        response = requests.get(nominatim_url, params=params, headers=headers)
        results = response.json()
        
        if not results:
            raise ValueError(f"Wilayah tidak ditemukan: {place_name}")
        
        result = results[0]
        
        if 'geojson' in result:
            geometry = shape(result['geojson'])
        else:
            # Create bounding box if no polygon
            bbox = result['boundingbox']
            geometry = box(
                float(bbox[2]), float(bbox[0]),
                float(bbox[3]), float(bbox[1])
            )
        
        gdf = gpd.GeoDataFrame(
            {'name': [result.get('display_name', place_name)]},
            geometry=[geometry],
            crs='EPSG:4326'
        )
        
        if buffer_km > 0:
            gdf_proj = gdf.to_crs(epsg=3857)
            gdf_proj['geometry'] = gdf_proj.geometry.buffer(buffer_km * 1000)
            gdf = gdf_proj.to_crs(epsg=4326)
        
        self.boundary_gdf = gdf
        self.boundary_polygon = gdf.geometry.unary_union
        self.area_name = place_name.replace(',', '_').replace(' ', '_')
        
        print(f"Place found via Nominatim!")
        print(f"  - Area: {self._calculate_area_km2():.2f} km²")
        
        return self.boundary_gdf
    
    def load_from_bbox(self, north: float, south: float, 
                      east: float, west: float,
                      name: str = "custom_area") -> gpd.GeoDataFrame:
        """
        Load boundary dari bounding box coordinates
        
        Parameters:
        -----------
        north, south, east, west : float
            Koordinat batas
        name : str
            Nama area
        """
        geometry = box(west, south, east, north)
        gdf = gpd.GeoDataFrame(
            {'name': [name]},
            geometry=[geometry],
            crs='EPSG:4326'
        )
        
        self.boundary_gdf = gdf
        self.boundary_polygon = geometry
        self.area_name = name
        
        print(f"Bounding box loaded!")
        print(f"  - Area: {self._calculate_area_km2():.2f} km²")
        
        return self.boundary_gdf
    
    def load_from_geojson(self, geojson_path: str) -> gpd.GeoDataFrame:
        """Load boundary dari file GeoJSON"""
        print(f"Loading GeoJSON: {geojson_path}")
        
        gdf = gpd.read_file(geojson_path)
        
        if gdf.crs is None or gdf.crs.to_epsg() != 4326:
            gdf = gdf.to_crs(epsg=4326)
        
        self.boundary_gdf = gdf
        self.boundary_polygon = gdf.geometry.unary_union
        self.area_name = Path(geojson_path).stem
        
        print(f"GeoJSON loaded successfully!")
        print(f"  - Area: {self._calculate_area_km2():.2f} km²")
        
        return self.boundary_gdf
    
    # ==================== DOWNLOAD METHODS ====================
    
    def download_layer(self, layer_type: OSMLayerType,
                      custom_tags: Optional[Dict] = None) -> Optional[gpd.GeoDataFrame]:
        """
        Download satu layer OSM
        
        Parameters:
        -----------
        layer_type : OSMLayerType
            Jenis layer yang akan didownload
        custom_tags : dict, optional
            Custom OSM tags (override default)
            
        Returns:
        --------
        GeoDataFrame dengan data OSM
        """
        if self.boundary_polygon is None:
            raise ValueError("Boundary belum di-load! Gunakan load_from_* method terlebih dahulu.")
        
        print(f"Downloading {layer_type.value}...")
        
        tags = custom_tags or OSM_TAGS.get(layer_type, {})
        
        for attempt in range(self.config.max_retries):
            try:
                if layer_type == OSMLayerType.ROADS:
                    # Special handling for road network
                    gdf = self._download_road_network()
                else:
                    # General features download
                    gdf = ox.features_from_polygon(
                        self.boundary_polygon,
                        tags=tags
                    )
                
                if gdf is not None and not gdf.empty:
                    # Clean up geometry
                    gdf = self._clean_geodataframe(gdf)
                    self.downloaded_data[layer_type.value] = gdf
                    
                    print(f"  - Downloaded {len(gdf)} features")
                    return gdf
                else:
                    print(f"  - No data found for {layer_type.value}")
                    return None
                    
            except Exception as e:
                print(f"  - Attempt {attempt + 1} failed: {str(e)[:100]}")
                if attempt < self.config.max_retries - 1:
                    time.sleep(self.config.retry_delay)
                else:
                    print(f"  - Failed to download {layer_type.value}")
                    return None
    
    def _download_road_network(self) -> gpd.GeoDataFrame:
        """Download road network dengan graph"""
        try:
            # Try to get graph first
            G = ox.graph_from_polygon(
                self.boundary_polygon,
                network_type='all',
                simplify=True
            )
            # Convert to GeoDataFrame
            gdf_nodes, gdf_edges = ox.graph_to_gdfs(G)
            return gdf_edges
        except Exception:
            # Fallback to features
            return ox.features_from_polygon(
                self.boundary_polygon,
                tags={'highway': True}
            )
    
    def download_all_layers(self, 
                           layers: Optional[List[OSMLayerType]] = None,
                           skip_errors: bool = True) -> Dict[str, gpd.GeoDataFrame]:
        """
        Download semua layer OSM yang dipilih
        
        Parameters:
        -----------
        layers : list, optional
            List layer yang akan didownload. Default: semua layer
        skip_errors : bool
            Skip layer yang error
            
        Returns:
        --------
        Dictionary dengan semua layer yang berhasil didownload
        """
        if layers is None:
            layers = [
                OSMLayerType.ROADS,
                OSMLayerType.BUILDINGS,
                OSMLayerType.WATER,
                OSMLayerType.LANDUSE,
                OSMLayerType.POI,
                OSMLayerType.NATURAL
            ]
        
        print(f"\nDownloading {len(layers)} layers for: {self.area_name}")
        print("=" * 50)
        
        results = {}
        
        for i, layer in enumerate(layers, 1):
            print(f"\n[{i}/{len(layers)}] ", end="")
            try:
                gdf = self.download_layer(layer)
                if gdf is not None:
                    results[layer.value] = gdf
            except Exception as e:
                print(f"  - Error: {str(e)[:100]}")
                if not skip_errors:
                    raise
        
        print("\n" + "=" * 50)
        print(f"Download complete! {len(results)}/{len(layers)} layers downloaded.")
        
        return results
    
    def download_custom(self, tags: Dict) -> Optional[gpd.GeoDataFrame]:
        """
        Download data dengan custom OSM tags
        
        Parameters:
        -----------
        tags : dict
            OSM tags untuk query
            Contoh: {'amenity': 'school'}
        """
        if self.boundary_polygon is None:
            raise ValueError("Boundary belum di-load!")
        
        print(f"Downloading custom features with tags: {tags}")
        
        try:
            gdf = ox.features_from_polygon(
                self.boundary_polygon,
                tags=tags
            )
            
            if not gdf.empty:
                gdf = self._clean_geodataframe(gdf)
                print(f"  - Downloaded {len(gdf)} features")
                return gdf
            else:
                print("  - No data found")
                return None
                
        except Exception as e:
            print(f"  - Error: {e}")
            return None
    
    # ==================== EXPORT METHODS ====================
    
    def export_layer(self, gdf: gpd.GeoDataFrame, 
                    layer_name: str,
                    export_format: ExportFormat = ExportFormat.GEOJSON,
                    output_dir: Optional[str] = None) -> str:
        """
        Export satu layer ke file
        
        Parameters:
        -----------
        gdf : GeoDataFrame
            Data yang akan di-export
        layer_name : str
            Nama layer untuk filename
        export_format : ExportFormat
            Format output
        output_dir : str, optional
            Directory output
            
        Returns:
        --------
        Path file yang di-export
        """
        output_dir = output_dir or self.config.output_dir
        area_dir = os.path.join(output_dir, self.area_name)
        Path(area_dir).mkdir(parents=True, exist_ok=True)
        
        # Clean filename
        safe_name = layer_name.replace(' ', '_').replace('/', '_')
        
        if export_format == ExportFormat.GEOJSON:
            filepath = os.path.join(area_dir, f"{safe_name}.geojson")
            gdf.to_file(filepath, driver='GeoJSON')
            
        elif export_format == ExportFormat.SHAPEFILE:
            filepath = os.path.join(area_dir, f"{safe_name}.shp")
            # Truncate column names for shapefile
            gdf_copy = gdf.copy()
            gdf_copy.columns = [c[:10] if len(c) > 10 else c for c in gdf_copy.columns]
            gdf_copy.to_file(filepath, driver='ESRI Shapefile')
            
        elif export_format == ExportFormat.GEOPACKAGE:
            filepath = os.path.join(area_dir, f"{safe_name}.gpkg")
            gdf.to_file(filepath, driver='GPKG')
            
        elif export_format == ExportFormat.CSV:
            filepath = os.path.join(area_dir, f"{safe_name}.csv")
            # Convert geometry to WKT for CSV
            gdf_copy = gdf.copy()
            gdf_copy['geometry_wkt'] = gdf_copy.geometry.apply(lambda g: g.wkt if g else None)
            gdf_copy = gdf_copy.drop(columns=['geometry'])
            gdf_copy.to_csv(filepath, index=False)
        
        print(f"  - Exported: {filepath}")
        return filepath
    
    def export_all(self, export_format: ExportFormat = ExportFormat.GEOJSON,
                  output_dir: Optional[str] = None) -> List[str]:
        """
        Export semua layer yang sudah di-download
        """
        if not self.downloaded_data:
            print("No data to export! Download data first.")
            return []
        
        print(f"\nExporting {len(self.downloaded_data)} layers as {export_format.value}...")
        
        exported_files = []
        
        for layer_name, gdf in self.downloaded_data.items():
            try:
                filepath = self.export_layer(gdf, layer_name, export_format, output_dir)
                exported_files.append(filepath)
            except Exception as e:
                print(f"  - Failed to export {layer_name}: {e}")
        
        # Also export boundary
        if self.boundary_gdf is not None:
            try:
                filepath = self.export_layer(self.boundary_gdf, 'boundary', export_format, output_dir)
                exported_files.append(filepath)
            except Exception as e:
                print(f"  - Failed to export boundary: {e}")
        
        print(f"\nExport complete! {len(exported_files)} files exported.")
        return exported_files
    
    # ==================== VISUALIZATION ====================
    
    def visualize_boundary(self) -> folium.Map:
        """
        Visualisasi boundary area dengan Folium
        """
        if self.boundary_gdf is None:
            raise ValueError("Boundary belum di-load!")
        
        # Get center
        centroid = self.boundary_polygon.centroid
        
        # Create map
        m = folium.Map(
            location=[centroid.y, centroid.x],
            zoom_start=11,
            tiles='cartodbpositron'
        )
        
        # Add boundary
        folium.GeoJson(
            self.boundary_gdf,
            style_function=lambda x: {
                'fillColor': '#3388ff',
                'color': '#3388ff',
                'weight': 2,
                'fillOpacity': 0.2
            },
            tooltip=f"Area: {self.area_name}"
        ).add_to(m)
        
        # Fit bounds
        bounds = self.boundary_gdf.total_bounds
        m.fit_bounds([[bounds[1], bounds[0]], [bounds[3], bounds[2]]])
        
        return m
    
    def visualize_all_layers(self) -> folium.Map:
        """
        Visualisasi semua layer yang sudah di-download
        """
        if self.boundary_gdf is None:
            raise ValueError("Boundary belum di-load!")
        
        centroid = self.boundary_polygon.centroid
        
        m = folium.Map(
            location=[centroid.y, centroid.x],
            zoom_start=12,
            tiles='cartodbpositron'
        )
        
        # Color scheme for layers
        colors = {
            'roads': '#e74c3c',
            'buildings': '#3498db',
            'water': '#2980b9',
            'landuse': '#27ae60',
            'poi': '#f39c12',
            'natural': '#16a085'
        }
        
        # Add boundary
        boundary_group = folium.FeatureGroup(name='Boundary')
        folium.GeoJson(
            self.boundary_gdf,
            style_function=lambda x: {
                'fillColor': 'transparent',
                'color': '#333',
                'weight': 3,
                'dashArray': '5, 5'
            }
        ).add_to(boundary_group)
        boundary_group.add_to(m)
        
        # Add each layer
        for layer_name, gdf in self.downloaded_data.items():
            if gdf is not None and not gdf.empty:
                layer_group = folium.FeatureGroup(name=layer_name.capitalize())
                
                color = colors.get(layer_name, '#9b59b6')
                
                # Sample if too many features
                if len(gdf) > 5000:
                    gdf_sample = gdf.sample(n=5000, random_state=42)
                else:
                    gdf_sample = gdf
                
                folium.GeoJson(
                    gdf_sample,
                    style_function=lambda x, c=color: {
                        'fillColor': c,
                        'color': c,
                        'weight': 1,
                        'fillOpacity': 0.5
                    }
                ).add_to(layer_group)
                
                layer_group.add_to(m)
        
        # Add layer control
        folium.LayerControl().add_to(m)
        
        return m
    
    def plot_static(self, figsize: Tuple[int, int] = (15, 10)):
        """
        Plot statis dengan matplotlib
        """
        if not self.downloaded_data:
            print("No data to plot!")
            return
        
        n_layers = len(self.downloaded_data)
        cols = min(3, n_layers)
        rows = (n_layers + cols - 1) // cols
        
        fig, axes = plt.subplots(rows, cols, figsize=figsize)
        if n_layers == 1:
            axes = [axes]
        else:
            axes = axes.flatten()
        
        colors = ['#e74c3c', '#3498db', '#27ae60', '#f39c12', '#9b59b6', '#16a085']
        
        for idx, (layer_name, gdf) in enumerate(self.downloaded_data.items()):
            ax = axes[idx]
            
            # Plot boundary
            if self.boundary_gdf is not None:
                self.boundary_gdf.boundary.plot(ax=ax, color='black', linewidth=2)
            
            # Plot layer
            if gdf is not None and not gdf.empty:
                gdf.plot(ax=ax, color=colors[idx % len(colors)], alpha=0.6)
            
            ax.set_title(f"{layer_name.capitalize()} ({len(gdf)} features)")
            ax.set_axis_off()
        
        # Hide unused axes
        for idx in range(len(self.downloaded_data), len(axes)):
            axes[idx].set_visible(False)
        
        plt.suptitle(f"OSM Data: {self.area_name}", fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()
    
    # ==================== HELPER METHODS ====================
    
    def _calculate_area_km2(self) -> float:
        """Calculate area in km²"""
        if self.boundary_gdf is None:
            return 0
        gdf_proj = self.boundary_gdf.to_crs(epsg=3857)
        return gdf_proj.geometry.area.sum() / 1e6
    
    def _clean_geodataframe(self, gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
        """Clean and standardize GeoDataFrame"""
        # Remove invalid geometries
        gdf = gdf[gdf.geometry.is_valid]
        
        # Remove empty geometries
        gdf = gdf[~gdf.geometry.is_empty]
        
        # Reset index
        gdf = gdf.reset_index(drop=True)
        
        # Convert list columns to string
        for col in gdf.columns:
            if gdf[col].apply(lambda x: isinstance(x, list)).any():
                gdf[col] = gdf[col].apply(lambda x: str(x) if isinstance(x, list) else x)
        
        return gdf
    
    def get_summary(self) -> pd.DataFrame:
        """Get summary of downloaded data"""
        summary_data = []
        
        for layer_name, gdf in self.downloaded_data.items():
            if gdf is not None:
                geom_types = gdf.geometry.geom_type.value_counts().to_dict()
                summary_data.append({
                    'Layer': layer_name,
                    'Features': len(gdf),
                    'Geometry Types': str(geom_types),
                    'Columns': len(gdf.columns)
                })
        
        return pd.DataFrame(summary_data)


print("OSMDownloader class loaded!")

## 5. Helper Functions

In [ ]:
def search_indonesian_region(query: str, max_results: int = 10) -> pd.DataFrame:
    """
    Cari wilayah Indonesia berdasarkan nama
    
    Parameters:
    -----------
    query : str
        Nama wilayah yang dicari
    max_results : int
        Jumlah maksimal hasil
        
    Returns:
    --------
    DataFrame dengan hasil pencarian
    """
    nominatim_url = "https://nominatim.openstreetmap.org/search"
    
    params = {
        'q': f"{query}, Indonesia",
        'format': 'json',
        'limit': max_results,
        'addressdetails': 1,
        'polygon_geojson': 0
    }
    
    headers = {'User-Agent': 'OSMDownloader/1.0'}
    
    response = requests.get(nominatim_url, params=params, headers=headers)
    results = response.json()
    
    if not results:
        print(f"Tidak ditemukan hasil untuk: {query}")
        return pd.DataFrame()
    
    data = []
    for r in results:
        data.append({
            'display_name': r.get('display_name', ''),
            'type': r.get('type', ''),
            'class': r.get('class', ''),
            'lat': float(r.get('lat', 0)),
            'lon': float(r.get('lon', 0)),
            'osm_id': r.get('osm_id', '')
        })
    
    return pd.DataFrame(data)


def get_indonesian_provinces() -> List[str]:
    """Daftar provinsi di Indonesia"""
    return [
        "Aceh", "Sumatera Utara", "Sumatera Barat", "Riau", "Kepulauan Riau",
        "Jambi", "Sumatera Selatan", "Kepulauan Bangka Belitung", "Bengkulu", "Lampung",
        "DKI Jakarta", "Banten", "Jawa Barat", "Jawa Tengah", "DI Yogyakarta", "Jawa Timur",
        "Bali", "Nusa Tenggara Barat", "Nusa Tenggara Timur",
        "Kalimantan Barat", "Kalimantan Tengah", "Kalimantan Selatan", "Kalimantan Timur", "Kalimantan Utara",
        "Sulawesi Utara", "Gorontalo", "Sulawesi Tengah", "Sulawesi Barat", "Sulawesi Selatan", "Sulawesi Tenggara",
        "Maluku", "Maluku Utara", "Papua", "Papua Barat", "Papua Selatan", "Papua Tengah", "Papua Pegunungan"
    ]


def batch_download_regions(regions: List[str], 
                          layers: List[OSMLayerType],
                          export_format: ExportFormat = ExportFormat.GEOJSON,
                          output_dir: str = "./osm_batch_output") -> Dict:
    """
    Download OSM data untuk multiple wilayah sekaligus
    
    Parameters:
    -----------
    regions : list
        List nama wilayah
    layers : list
        List layer yang akan didownload
    export_format : ExportFormat
        Format export
    output_dir : str
        Directory output
        
    Returns:
    --------
    Dictionary dengan status setiap wilayah
    """
    results = {}
    
    for i, region in enumerate(regions, 1):
        print(f"\n{'='*60}")
        print(f"Processing [{i}/{len(regions)}]: {region}")
        print(f"{'='*60}")
        
        try:
            downloader = OSMDownloader()
            downloader.load_from_place_name(f"{region}, Indonesia")
            downloader.download_all_layers(layers)
            downloader.export_all(export_format, output_dir)
            
            results[region] = {
                'status': 'success',
                'layers': list(downloader.downloaded_data.keys()),
                'features': sum(len(gdf) for gdf in downloader.downloaded_data.values())
            }
            
        except Exception as e:
            results[region] = {
                'status': 'failed',
                'error': str(e)
            }
        
        # Rate limiting
        time.sleep(2)
    
    return results


print("Helper functions loaded!")

## 6. Interactive Widget Interface

In [ ]:
class OSMDownloaderUI:
    """
    Interactive UI untuk OSM Downloader menggunakan ipywidgets
    """
    
    def __init__(self):
        self.downloader = None
        self.setup_widgets()
    
    def setup_widgets(self):
        """Setup all widgets"""
        
        # Input method selection
        self.input_method = widgets.RadioButtons(
            options=['Nama Kabupaten/Kota', 'Shapefile (.shp)', 'GeoJSON', 'Bounding Box'],
            value='Nama Kabupaten/Kota',
            description='Input Method:',
            style={'description_width': '120px'}
        )
        
        # Place name input
        self.place_name = widgets.Text(
            value='',
            placeholder='Contoh: Kota Bandung atau Kabupaten Bogor',
            description='Nama Wilayah:',
            style={'description_width': '120px'},
            layout=widgets.Layout(width='500px')
        )
        
        # File path input
        self.file_path = widgets.Text(
            value='',
            placeholder='Path ke file .shp atau .geojson',
            description='File Path:',
            style={'description_width': '120px'},
            layout=widgets.Layout(width='500px')
        )
        
        # Bounding box inputs
        self.bbox_north = widgets.FloatText(value=-6.0, description='North:', style={'description_width': '60px'})
        self.bbox_south = widgets.FloatText(value=-7.0, description='South:', style={'description_width': '60px'})
        self.bbox_east = widgets.FloatText(value=108.0, description='East:', style={'description_width': '60px'})
        self.bbox_west = widgets.FloatText(value=107.0, description='West:', style={'description_width': '60px'})
        
        # Layer selection
        self.layer_checkboxes = widgets.SelectMultiple(
            options=['roads', 'buildings', 'water', 'landuse', 'poi', 'natural', 'railway', 'amenity'],
            value=['roads', 'buildings'],
            description='Layers:',
            style={'description_width': '120px'},
            layout=widgets.Layout(height='150px', width='300px')
        )
        
        # Export format
        self.export_format = widgets.Dropdown(
            options=['geojson', 'shp', 'gpkg', 'csv'],
            value='geojson',
            description='Export Format:',
            style={'description_width': '120px'}
        )
        
        # Buffer
        self.buffer_km = widgets.FloatSlider(
            value=0,
            min=0,
            max=10,
            step=0.5,
            description='Buffer (km):',
            style={'description_width': '120px'}
        )
        
        # Output directory
        self.output_dir = widgets.Text(
            value='./osm_output',
            description='Output Dir:',
            style={'description_width': '120px'},
            layout=widgets.Layout(width='400px')
        )
        
        # Buttons
        self.search_btn = widgets.Button(
            description='Search Region',
            button_style='info',
            icon='search'
        )
        
        self.load_btn = widgets.Button(
            description='Load Boundary',
            button_style='primary',
            icon='map'
        )
        
        self.download_btn = widgets.Button(
            description='Download OSM Data',
            button_style='success',
            icon='download'
        )
        
        self.export_btn = widgets.Button(
            description='Export Data',
            button_style='warning',
            icon='save'
        )
        
        # Output area
        self.output = widgets.Output()
        self.map_output = widgets.Output()
        
        # Event handlers
        self.search_btn.on_click(self.on_search)
        self.load_btn.on_click(self.on_load)
        self.download_btn.on_click(self.on_download)
        self.export_btn.on_click(self.on_export)
        self.input_method.observe(self.on_method_change, names='value')
    
    def on_method_change(self, change):
        """Handle input method change"""
        pass
    
    def on_search(self, btn):
        """Handle search button click"""
        with self.output:
            clear_output()
            query = self.place_name.value
            if query:
                results = search_indonesian_region(query)
                if not results.empty:
                    display(results)
                else:
                    print("Tidak ada hasil ditemukan")
            else:
                print("Masukkan nama wilayah terlebih dahulu")
    
    def on_load(self, btn):
        """Handle load button click"""
        with self.output:
            clear_output()
            self.downloader = OSMDownloader(OSMConfig(output_dir=self.output_dir.value))
            
            method = self.input_method.value
            buffer = self.buffer_km.value
            
            try:
                if method == 'Nama Kabupaten/Kota':
                    place = self.place_name.value
                    if not place:
                        print("Masukkan nama wilayah!")
                        return
                    self.downloader.load_from_place_name(f"{place}, Indonesia", buffer_km=buffer)
                    
                elif method == 'Shapefile (.shp)':
                    path = self.file_path.value
                    if not path:
                        print("Masukkan path file!")
                        return
                    self.downloader.load_from_shapefile(path, buffer_km=buffer)
                    
                elif method == 'GeoJSON':
                    path = self.file_path.value
                    if not path:
                        print("Masukkan path file!")
                        return
                    self.downloader.load_from_geojson(path)
                    
                elif method == 'Bounding Box':
                    self.downloader.load_from_bbox(
                        self.bbox_north.value,
                        self.bbox_south.value,
                        self.bbox_east.value,
                        self.bbox_west.value
                    )
                
                # Show map
                with self.map_output:
                    clear_output()
                    m = self.downloader.visualize_boundary()
                    display(m)
                    
            except Exception as e:
                print(f"Error: {e}")
    
    def on_download(self, btn):
        """Handle download button click"""
        with self.output:
            clear_output()
            
            if self.downloader is None or self.downloader.boundary_polygon is None:
                print("Load boundary terlebih dahulu!")
                return
            
            layers = [OSMLayerType(l) for l in self.layer_checkboxes.value]
            
            if not layers:
                print("Pilih minimal satu layer!")
                return
            
            self.downloader.download_all_layers(layers)
            
            # Show summary
            print("\nSummary:")
            display(self.downloader.get_summary())
            
            # Update map
            with self.map_output:
                clear_output()
                m = self.downloader.visualize_all_layers()
                display(m)
    
    def on_export(self, btn):
        """Handle export button click"""
        with self.output:
            if self.downloader is None or not self.downloader.downloaded_data:
                print("Download data terlebih dahulu!")
                return
            
            fmt = ExportFormat(self.export_format.value)
            self.downloader.export_all(fmt)
    
    def display(self):
        """Display the UI"""
        
        # Title
        title = widgets.HTML("<h2>OSM Map Downloader - Indonesia</h2>")
        
        # Input section based on method
        place_input = widgets.VBox([
            self.place_name,
            widgets.HBox([self.search_btn])
        ])
        
        file_input = widgets.VBox([self.file_path])
        
        bbox_input = widgets.VBox([
            widgets.HBox([self.bbox_north, self.bbox_south]),
            widgets.HBox([self.bbox_west, self.bbox_east])
        ])
        
        # Main layout
        input_section = widgets.VBox([
            widgets.HTML("<h4>1. Pilih Input Method & Load Boundary</h4>"),
            self.input_method,
            place_input,
            file_input,
            bbox_input,
            self.buffer_km,
            self.load_btn
        ])
        
        download_section = widgets.VBox([
            widgets.HTML("<h4>2. Pilih Layer & Download</h4>"),
            self.layer_checkboxes,
            self.download_btn
        ])
        
        export_section = widgets.VBox([
            widgets.HTML("<h4>3. Export Data</h4>"),
            self.export_format,
            self.output_dir,
            self.export_btn
        ])
        
        # Combine sections
        controls = widgets.HBox([
            input_section,
            download_section,
            export_section
        ], layout=widgets.Layout(justify_content='space-around'))
        
        # Full layout
        full_layout = widgets.VBox([
            title,
            controls,
            widgets.HTML("<hr>"),
            widgets.HTML("<h4>Output:</h4>"),
            self.output,
            widgets.HTML("<h4>Map Preview:</h4>"),
            self.map_output
        ])
        
        display(full_layout)


print("Interactive UI loaded!")

---
# CONTOH PENGGUNAAN
---

## Contoh 1: Download menggunakan Nama Kabupaten/Kota

In [ ]:
# Inisialisasi downloader
downloader = OSMDownloader()

# Load boundary dari nama wilayah
# Contoh: Kota Bandung, Kabupaten Bogor, Kota Surabaya, dll
downloader.load_from_place_name("Kota Bandung, Indonesia")

In [ ]:
# Visualisasi boundary
downloader.visualize_boundary()

In [ ]:
# Download beberapa layer
layers_to_download = [
    OSMLayerType.ROADS,
    OSMLayerType.BUILDINGS,
    OSMLayerType.WATER,
    OSMLayerType.POI
]

downloader.download_all_layers(layers_to_download)

In [ ]:
# Lihat summary
downloader.get_summary()

In [ ]:
# Visualisasi semua layer
downloader.visualize_all_layers()

In [ ]:
# Export ke GeoJSON
downloader.export_all(ExportFormat.GEOJSON)

In [ ]:
# Export ke Shapefile
downloader.export_all(ExportFormat.SHAPEFILE)

## Contoh 2: Download menggunakan Shapefile

In [ ]:
# Inisialisasi downloader baru
downloader_shp = OSMDownloader()

# Load dari shapefile (ganti dengan path shapefile Anda)
# downloader_shp.load_from_shapefile("path/to/your/shapefile.shp", buffer_km=1)

# Contoh dengan dummy shapefile
print("Untuk menggunakan shapefile, uncomment kode di atas dan ganti path-nya")
print("Contoh:")
print('  downloader_shp.load_from_shapefile("./data/batas_wilayah.shp")')

## Contoh 3: Download Data Spesifik dengan Custom Tags

In [ ]:
# Download hanya sekolah
schools = downloader.download_custom({'amenity': 'school'})

if schools is not None:
    print(f"Ditemukan {len(schools)} sekolah")
    display(schools[['name', 'geometry']].head(10) if 'name' in schools.columns else schools.head(10))

In [ ]:
# Download rumah sakit
hospitals = downloader.download_custom({'amenity': 'hospital'})

if hospitals is not None:
    print(f"Ditemukan {len(hospitals)} rumah sakit")

In [ ]:
# Download tempat ibadah
worship_places = downloader.download_custom({'amenity': 'place_of_worship'})

if worship_places is not None:
    print(f"Ditemukan {len(worship_places)} tempat ibadah")

## Contoh 4: Batch Download Multiple Wilayah

In [ ]:
# Daftar wilayah yang akan didownload
regions = [
    "Kota Bandung",
    "Kota Cimahi",
    # "Kabupaten Bandung",
    # Tambahkan wilayah lain sesuai kebutuhan
]

# Layer yang akan didownload
layers = [
    OSMLayerType.ROADS,
    OSMLayerType.BUILDINGS
]

# Batch download (uncomment untuk menjalankan)
# results = batch_download_regions(regions, layers, ExportFormat.GEOJSON)
# print("\nBatch Download Results:")
# for region, status in results.items():
#     print(f"  {region}: {status['status']}")

## Contoh 5: Cari Wilayah Indonesia

In [ ]:
# Cari wilayah berdasarkan nama
search_indonesian_region("Bandung")

In [ ]:
# Cari kabupaten di Jawa Barat
search_indonesian_region("Kabupaten Jawa Barat")

## Contoh 6: Download dengan Bounding Box

In [ ]:
# Inisialisasi downloader
downloader_bbox = OSMDownloader()

# Load dari bounding box
# Contoh: area sekitar Monas, Jakarta
downloader_bbox.load_from_bbox(
    north=-6.16,
    south=-6.18,
    east=106.84,
    west=106.82,
    name="Monas_Area"
)

# Visualisasi
downloader_bbox.visualize_boundary()

In [ ]:
# Download data
downloader_bbox.download_all_layers([OSMLayerType.ROADS, OSMLayerType.BUILDINGS])
downloader_bbox.visualize_all_layers()

## Contoh 7: Interactive UI (Widget)

In [ ]:
# Jalankan Interactive UI
# Catatan: Membutuhkan ipywidgets yang ter-enable di Jupyter

try:
    ui = OSMDownloaderUI()
    ui.display()
except Exception as e:
    print(f"Widget tidak tersedia: {e}")
    print("Gunakan contoh code-based di atas sebagai alternatif.")

## Contoh 8: Plot Statis dengan Matplotlib

In [ ]:
# Plot statis dari data yang sudah didownload
if downloader.downloaded_data:
    downloader.plot_static(figsize=(12, 8))

---
## Tips & Catatan

### OSM Tags yang Sering Digunakan:

| Kategori | Tag | Deskripsi |
|----------|-----|------------|
| Jalan | `highway` | Semua jenis jalan |
| Bangunan | `building` | Semua bangunan |
| Air | `water`, `waterway` | Sungai, danau, dll |
| Sekolah | `amenity=school` | Sekolah |
| Rumah Sakit | `amenity=hospital` | Rumah sakit |
| Masjid | `amenity=place_of_worship`, `religion=muslim` | Masjid |
| Gereja | `amenity=place_of_worship`, `religion=christian` | Gereja |
| SPBU | `amenity=fuel` | Pom bensin |
| ATM | `amenity=atm` | ATM |
| Bank | `amenity=bank` | Bank |
| Restoran | `amenity=restaurant` | Restoran |
| Cafe | `amenity=cafe` | Cafe |
| Supermarket | `shop=supermarket` | Supermarket |
| Pasar | `amenity=marketplace` | Pasar |

### Referensi OSM Tags:
- https://wiki.openstreetmap.org/wiki/Map_Features
- https://taginfo.openstreetmap.org/

### Limitasi:
1. Area yang terlalu besar mungkin timeout - gunakan buffer yang sesuai
2. Rate limiting dari Nominatim - jangan query terlalu sering
3. Data OSM bergantung pada kontribusi komunitas - coverage bervariasi
---

In [ ]:
print("Notebook selesai! Gunakan contoh-contoh di atas untuk mengunduh data OSM.")